# Experience scoring

Use the cleaned IMDb reviews to build experience profiles for each movie and show.

The 55 experience attributes are represented by short descriptions, and a pretrained sentence embedding model is used to compare those descriptions with the language in audience reviews.

In [1]:
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.dataset as ds
import pyarrow.parquet as pq

from sentence_transformers import SentenceTransformer


# Get the main project folder from the notebooks folder
project_root = Path.cwd().parent

# Clean reviews created in notebook 04
clean_reviews_dir = (
    project_root
    / "data"
    / "processed"
    / "clean_reviews"
)

# Treat all cleaned review files as one dataset
review_dataset = ds.dataset(
    clean_reviews_dir,
    format="parquet"
)
print(f"Reviews available: {review_dataset.count_rows():,}")


C:\Users\dorca\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Reviews available: 2,723,221


## Set up the experience taxonomy

Define the 55 viewing-experience attributes and the descriptions used for semantic matching.

In [2]:
# Experience taxonomy
experience_taxonomy = {
    "Emotion": {
        "Happy": "Produces a generally happy or positive feeling.",
        "Comforting": "Feels safe, soothing, or emotionally comforting.",
        "Bittersweet": "Mixes happiness or warmth with sadness or loss.",
        "Cute": "Feels sweet, adorable, or endearing.",
        "Stomach butterflies": "Creates giddy excitement, often romantic.",
        "Hopeful": "Creates or restores a sense that things can get better.",
        "Heart-wrenching": "Produces strong emotional pain or heartbreak.",
        "Warm / tender": "Feels emotionally affectionate, gentle, or tender.",
        "Yearning / longing": "Creates a strong feeling of wanting someone or something.",
        "Sad / melancholic": "Creates sadness, melancholy, or a lingering low mood.",
        "Romantic": "Creates a strong sense of romance, love, or romantic feeling.",
    },

    "Overall experience": {
        "Full of life": "Feels vibrant, alive, and rich with human experience.",
        "Fun": "Primarily feels enjoyable and entertaining.",
        "Intense": "Produces unusually strong emotion or sensation.",
        "Life-affirming": "Leaves you appreciating life, people, or existence.",
        "Escapist": "Lets you mentally disappear from ordinary life.",
        "Immersive": "Strongly absorbs you into its story or world.",
        "Haunting": "Lingers with you in an eerie or emotionally unsettling way.",
    },

    "Thinking / engagement": {
        "Existential": "Makes you contemplate existence, identity, mortality, or meaning.",
        "Reflective / thought-provoking": "Makes you think deeply or reconsider something.",
        "Provokes curiosity": "Makes you strongly want to understand or discover more.",
        "Unpredictable": "Makes it difficult to anticipate what will happen.",
        "Suspenseful / tense": "Creates anticipation, nervousness, or edge-of-your-seat tension.",
    },

    "Tone & style": {
        "Funny": "Produces laughter or amusement.",
        "Dark humor": "Finds humor in dark, serious, taboo, or disturbing situations.",
        "Quirky": "Feels unconventional in a playful or charming way.",
        "Bizarre": "Feels strange, unusual, surreal, or WTF-inducing.",
        "Dark": "Has a bleak, disturbing, sinister, or heavy tone.",
        "Gruesome": "Feels viscerally graphic, bloody, or disturbing.",
        "Scary / frightening": "Creates fear, fright, or a strong sense of being scared.",
    },

    "Story / world experience": {
        "Adventurous": "Creates excitement through exploration, journeys, challenges, or discovery.",
        "Epic / grand": "Feels large in scale, world, conflict, or ambition.",
        "High-stakes": "Makes the consequences feel particularly significant.",
        "Morally complex / gray": "Complicates simple ideas of good and bad or right and wrong.",
        "Ruthless / unforgiving": "Creates a sense that consequences are real and characters are not protected.",
    },

    "Connection": {
        "Real / relatable": "Feels recognizable to real experiences, emotions, or relationships.",
        "Human / humanistic": "Captures the messiness, contradictions, and complexity of being human.",
        "Familiar": "Produces a sense of recognition or familiarity.",
        "Strong chemistry": "Characters have a compelling interpersonal dynamic.",
        "Nostalgic": "Evokes an emotional connection to or longing for the past.",
    },

    "Viewing style": {
        "Easy to watch": "Requires relatively little mental or emotional effort to follow and enjoy.",
        "Fast-paced": "Moves quickly with relatively little downtime.",
        "Slow-burn": "Develops deliberately and rewards patience.",
        "Rewatchable": "Has qualities that make repeat viewing particularly appealing.",
    },

    "Atmosphere": {
        "Seasonal": "Strongly evokes a particular season such as fall, winter, spring, or summer.",
    },

    "Themes / lens": {
        "Feminist": "Meaningfully engages with women, gender, power, or feminist ideas.",
        "Political": "Meaningfully engages with politics, institutions, ideology, or power.",
        "Socially critical": "Critiques or questions some aspect of society, culture, or institutions.",
    },

    "After-effect": {
        "Unforgettable": "Leaves an unusually strong lasting impression.",
        "Peaceful": "Leaves you feeling calm or settled.",
        "Fulfilled": "Leaves you emotionally satisfied.",
        "Devastated": "Leaves you emotionally wrecked.",
        "Unsettled": "Leaves lingering discomfort or unease.",
        "Inspired / motivated": "Leaves you wanting to act, create, change, or pursue something.",
        "Cathartic": "Creates a sense of emotional release after strong or difficult feelings.",
    },
}

attribute_count = sum(
    len(attributes)
    for attributes in experience_taxonomy.values()
)

print(f"Categories: {len(experience_taxonomy)}")
print(f"Experience attributes: {attribute_count}")

Categories: 10
Experience attributes: 55


## Choose the embedding model

Compare two lightweight sentence embedding models before choosing the model for the full review-scoring run.

In [3]:
# Start with MiniLM as a lightweight baseline
minilm_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

attribute_names = []
attribute_categories = []
attribute_descriptions = []

for category, attributes in experience_taxonomy.items():
    for attribute, description in attributes.items():
        attribute_categories.append(category)
        attribute_names.append(attribute)

        # Include both the name and definition in the text sent to the model
        attribute_descriptions.append(
            f"{attribute}. {description}"
        )

# Embed the 55 definitions with MiniLM
minilm_attribute_embeddings = minilm_model.encode(
    attribute_descriptions,
    normalize_embeddings=True
)

print("Baseline model:", "sentence-transformers/all-MiniLM-L6-v2")
print("Attribute embeddings:", minilm_attribute_embeddings.shape)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2432.82it/s]


Baseline model: sentence-transformers/all-MiniLM-L6-v2
Attribute embeddings: (55, 384)


### MiniLM baseline

Test MiniLM on a review with several clear emotional signals and inspect its strongest experience matches.

In [4]:
test_review = """
This was such a gentle and emotional movie. It felt like a warm hug,
but there was also this sadness underneath everything. I cried at the
ending, but somehow came away feeling hopeful about life.
"""

# Embed the test review using the same model
review_embedding = minilm_model.encode(
    [test_review],
    normalize_embeddings=True
)

# Compare the review with all 55 experience definitions
similarities = review_embedding @ minilm_attribute_embeddings.T

# Get the 10 strongest matches
top_indices = np.argsort(similarities[0])[::-1][:10]

test_results = pd.DataFrame({
    "category": [
        attribute_categories[i]
        for i in top_indices
    ],
    "experience": [
        attribute_names[i]
        for i in top_indices
    ],
    "similarity": [
        similarities[0][i]
        for i in top_indices
    ],
})

test_results

,category,experience,similarity
0,Emotion,Bittersweet,0.420808
1,Emotion,Sad / melancholic,0.409888
2,Emotion,Comforting,0.401952
3,Emotion,Heart-wrenching,0.391779
4,Emotion,Cute,0.379216
5,Tone & style,Gruesome,0.375806
6,After-effect,Devastated,0.373955
7,After-effect,Unsettled,0.371949
8,Overall experience,Haunting,0.368101
9,Emotion,Yearning / longing,0.359987


### Compare with BGE

MiniLM captured the general emotional tone but also produced some unrelated matches. Test BGE on the same review to see whether the semantic matches are more coherent.

In [5]:
# Load the model selected for the full scoring run
bge_model = SentenceTransformer(
    "BAAI/bge-small-en-v1.5"
)

# Match the configuration used during GPU scoring
bge_model.max_seq_length = 128

print("Model:", "BAAI/bge-small-en-v1.5")
print("Max sequence length:", bge_model.max_seq_length)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2777.60it/s]


Model: BAAI/bge-small-en-v1.5
Max sequence length: 128


In [6]:
# Embed the same 55 definitions with the BGE model
bge_attribute_embeddings = bge_model.encode(
    attribute_descriptions,
    normalize_embeddings=True
)

# Embed the same test review
bge_review_embedding = bge_model.encode(
    [test_review],
    normalize_embeddings=True
)

# Compare the review with all 55 attributes
bge_similarities = (
    bge_review_embedding
    @ bge_attribute_embeddings.T
)

# Show the strongest matches
top_indices = np.argsort(
    bge_similarities[0]
)[::-1][:10]

bge_test_results = pd.DataFrame({
    "category": [
        attribute_categories[i]
        for i in top_indices
    ],
    "experience": [
        attribute_names[i]
        for i in top_indices
    ],
    "similarity": [
        bge_similarities[0][i]
        for i in top_indices
    ],
})

bge_test_results

,category,experience,similarity
0,Emotion,Warm / tender,0.752587
1,Emotion,Bittersweet,0.702600
2,Connection,Real / relatable,0.688191
3,Emotion,Comforting,0.684566
4,Emotion,Cute,0.682557
5,After-effect,Fulfilled,0.682509
6,Emotion,Heart-wrenching,0.682295
7,Emotion,Yearning / longing,0.680624
8,After-effect,Cathartic,0.679026
9,After-effect,Peaceful,0.675255


## Benchmark BGE locally

BGE produced the more coherent semantic matches, so use it for the scoring stage. Benchmark it on real review text first to see whether processing the full dataset locally is practical.

In [7]:
# Pull a small sample of real reviews for the speed test
benchmark_table = review_dataset.head(
    1000,
    columns=["review_id", "review_text"]
)

benchmark_reviews = benchmark_table.to_pandas()

print(f"Reviews in benchmark: {len(benchmark_reviews):,}")
benchmark_reviews.head(3)

Reviews in benchmark: 1,000


,review_id,review_text
0,rw5704482,Very Strong Season 2. I enjoyed the first seas...
1,rw5704483,Icelandic detectives?. I know Iceland is a sma...
2,rw5704484,"Nothing special. Except K K , no other actor l..."


In [8]:
# Time how long BGE takes to embed 1,000 real reviews
start = perf_counter()

benchmark_embeddings = bge_model.encode(
    benchmark_reviews["review_text"].tolist(),
    batch_size=32,
    normalize_embeddings=True,
    show_progress_bar=True
)

elapsed = perf_counter() - start
reviews_per_second = len(benchmark_reviews) / elapsed

print(f"Embedding shape: {benchmark_embeddings.shape}")
print(f"Time: {elapsed:.1f} seconds")
print(f"Speed: {reviews_per_second:.1f} reviews/second")

estimated_hours = (
    review_dataset.count_rows()
    / reviews_per_second
    / 3600
)

print(
    f"Estimated full local run: "
    f"{estimated_hours:.1f} hours"
)

Batches: 100%|██████████| 32/32 [01:02<00:00,  1.95s/it]

Embedding shape: (1000, 384)
Time: 62.4 seconds
Speed: 16.0 reviews/second
Estimated full local run: 47.2 hours


## Prepare the scoring input

Create the dataset used for GPU scoring. Movies and TV shows can share the same numeric TMDB ID, so `media_type` and `tmdb_id` are combined into a unique title key before scoring.

In [9]:
# Build the scoring dataset with a unique title key
scoring_input_dir = (
    project_root
    / "data"
    / "processed"
    / "scoring_input"
)

scoring_input_dir.mkdir(
    parents=True,
    exist_ok=True
)

# Clear partial output if this cell is rerun
for old_file in scoring_input_dir.glob("*.parquet"):
    old_file.unlink()

scanner = review_dataset.scanner(
    columns=[
        "review_id",
        "tmdb_id",
        "media_type",
        "review_text"
    ],
    batch_size=100_000
)

rows_per_file = 250_000
rows_written = 0
rows_in_file = 0
file_number = 0
writer = None

for batch in scanner.to_batches():

    batch_df = batch.to_pandas()

    # TMDB IDs are identifiers, so keep them as integers
    batch_df["tmdb_id"] = (
        batch_df["tmdb_id"]
        .astype("int64")
    )

    # Movie and TV IDs can overlap, so use both as the title key
    batch_df["title_key"] = (
        batch_df["media_type"]
        + ":"
        + batch_df["tmdb_id"].astype(str)
    )

    batch_df = batch_df[
        [
            "review_id",
            "title_key",
            "tmdb_id",
            "media_type",
            "review_text"
        ]
    ]

    start = 0

    while start < len(batch_df):

        # Fill the current output file up to 250,000 rows
        space_left = rows_per_file - rows_in_file
        chunk_df = batch_df.iloc[start:start + space_left]

        table = pa.Table.from_pandas(
            chunk_df,
            preserve_index=False
        )

        if writer is None:
            output_file = (
                scoring_input_dir
                / f"part-{file_number}.parquet"
            )

            writer = pq.ParquetWriter(
                output_file,
                table.schema,
                compression="zstd"
            )

        writer.write_table(table)

        chunk_rows = len(chunk_df)
        rows_in_file += chunk_rows
        rows_written += chunk_rows
        start += chunk_rows

        # Close the file once it reaches 250,000 reviews
        if rows_in_file == rows_per_file:
            writer.close()
            writer = None

            print(
                f"File {file_number + 1} written | "
                f"{rows_written:,} reviews"
            )

            file_number += 1
            rows_in_file = 0

# Close the final smaller file
if writer is not None:
    writer.close()

    print(
        f"File {file_number + 1} written | "
        f"{rows_written:,} reviews"
    )

print()
print(f"Total reviews written: {rows_written:,}")

File 1 written | 250,000 reviews
File 2 written | 500,000 reviews
File 3 written | 750,000 reviews
File 4 written | 1,000,000 reviews
File 5 written | 1,250,000 reviews
File 6 written | 1,500,000 reviews
File 7 written | 1,750,000 reviews
File 8 written | 2,000,000 reviews
File 9 written | 2,250,000 reviews
File 10 written | 2,500,000 reviews
File 11 written | 2,723,221 reviews

Total reviews written: 2,723,221


### Validate the scoring input

Confirm that every cleaned review is present, the movie and TV title keys are represented correctly, and the transfer dataset is split into the expected number of Parquet files.

In [10]:
scoring_dataset = ds.dataset(
    scoring_input_dir,
    format="parquet"
)

scoring_files = sorted(
    scoring_input_dir.glob("*.parquet")
)

size_gb = sum(
    file.stat().st_size
    for file in scoring_files
) / (1024 ** 3)

key_check = (
    scoring_dataset
    .to_table(columns=["title_key"])
    .to_pandas()["title_key"]
    .nunique()
)

print(f"Reviews:           {scoring_dataset.count_rows():,}")
print(f"Parquet files:     {len(scoring_files):,}")
print(f"Unique title keys: {key_check:,}")
print(f"Size:              {size_gb:.2f} GB")

Reviews:           2,723,221
Parquet files:     11
Unique title keys: 7,721
Size:              1.25 GB


## Run semantic scoring on GPU

The local benchmark showed that embedding the full review dataset on CPU would take too long, so the production scoring step was run on a Tesla T4 GPU in Google Colab.

The final model was `BAAI/bge-small-en-v1.5`. Every cleaned review contributed to scoring, with each review limited to a maximum model sequence length of 128 tokens to make the full run practical.

For each review:

1. Encode the review text with BGE.
2. Compare the review embedding with the 55 normalized experience-definition embeddings using cosine similarity.
3. Add the 55 similarity values to the running totals for that review's `title_key`.
4. Track how many reviews contributed to each title.

The 11 scoring-input Parquet files were processed separately and saved as checkpoint files. Only the title-level similarity sums and review counts were retained, rather than storing millions of review embeddings.

## Build the final experience scores

Combine the GPU scoring checkpoints and calculate the average semantic similarity for each title and experience.

### Load the GPU checkpoints

Bring the title-level aggregates from the GPU run back into the local pipeline and combine titles that appeared across multiple input files.

In [11]:
checkpoint_dir = (
    project_root
    / "data"
    / "processed"
    / "experience_checkpoints"
)

# Use the exact 11 checkpoint files from the GPU run
checkpoint_files = [
    checkpoint_dir / f"checkpoint_{i:02d}.parquet"
    for i in range(1, 12)
]

missing_files = [
    file.name
    for file in checkpoint_files
    if not file.exists()
]

if missing_files:
    raise FileNotFoundError(
        f"Missing checkpoint files: {missing_files}"
    )

print(f"Checkpoint files: {len(checkpoint_files)}")

for file in checkpoint_files:
    print(file.name)

Checkpoint files: 11
checkpoint_01.parquet
checkpoint_02.parquet
checkpoint_03.parquet
checkpoint_04.parquet
checkpoint_05.parquet
checkpoint_06.parquet
checkpoint_07.parquet
checkpoint_08.parquet
checkpoint_09.parquet
checkpoint_10.parquet
checkpoint_11.parquet


In [12]:
checkpoint_df = pd.concat(
    [
        pd.read_parquet(file)
        for file in checkpoint_files
    ],
    ignore_index=True
)

sum_columns = [
    f"similarity_sum_{i:02d}"
    for i in range(55)
]

aggregation = {
    "tmdb_id": "first",
    "media_type": "first",
    "reviews_used": "sum"
}

for column in sum_columns:
    aggregation[column] = "sum"

# Combine titles that appeared across multiple input files
title_scores = (
    checkpoint_df
    .groupby("title_key", as_index=False)
    .agg(aggregation)
)

print(f"Titles:              {len(title_scores):,}")
print(f"Reviews represented: {title_scores['reviews_used'].sum():,}")

# Make sure the complete scoring run is represented
assert len(title_scores) == 7_721
assert title_scores["reviews_used"].sum() == 2_723_221

print("Checkpoint validation passed.")

Titles:              7,721
Reviews represented: 2,723,221
Checkpoint validation passed.


### Calculate title-level similarities

Divide each title's accumulated similarity totals by its review count to get the average semantic similarity for all 55 experiences.

In [13]:
# Calculate all 55 average similarities at once
raw_similarity_columns = [
    f"raw_similarity_{i:02d}"
    for i in range(55)
]

raw_similarities = (
    title_scores[sum_columns]
    .div(
        title_scores["reviews_used"],
        axis=0
    )
)

raw_similarities.columns = raw_similarity_columns

# Remove old similarity columns if this cell is rerun
existing_raw_columns = [
    column
    for column in raw_similarity_columns
    if column in title_scores.columns
]

if existing_raw_columns:
    title_scores = title_scores.drop(
        columns=existing_raw_columns
    )

title_scores = pd.concat(
    [title_scores, raw_similarities],
    axis=1
)

print("Average experience similarities calculated.")
print(
    "Duplicate column names:",
    title_scores.columns.duplicated().sum()
)

Average experience similarities calculated.
Duplicate column names: 0


## Build the title experience profiles

Reshape the 55 similarity values into one row per title and experience, then convert the raw similarities into relative 0–100 scores.

In [14]:
# Rebuild the experience lookup directly from the taxonomy
experience_pairs = [
    (category, experience)
    for category, experiences in experience_taxonomy.items()
    for experience in experiences
]

print("Experiences in taxonomy:", len(experience_pairs))
print("Unique experience names:", len(set(
    experience for _, experience in experience_pairs
)))

Experiences in taxonomy: 55
Unique experience names: 55


In [15]:
experience_lookup = pd.DataFrame(
    {
        "attribute_index": range(55),
        "category": [
            category
            for category, experience in experience_pairs
        ],
        "experience": [
            experience
            for category, experience in experience_pairs
        ]
    }
)

# Reshape the title-level similarities into long format
experience_scores = (
    title_scores[
        [
            "title_key",
            "tmdb_id",
            "media_type",
            "reviews_used"
        ]
        + raw_similarity_columns
    ]
    .melt(
        id_vars=[
            "title_key",
            "tmdb_id",
            "media_type",
            "reviews_used"
        ],
        var_name="score_column",
        value_name="raw_similarity"
    )
)

experience_scores["attribute_index"] = (
    experience_scores["score_column"]
    .str.extract(r"(\d+)$")[0]
    .astype(int)
)

experience_scores = (
    experience_scores
    .merge(
        experience_lookup,
        on="attribute_index",
        how="left",
        validate="many_to_one"
    )
    .drop(
        columns=[
            "score_column",
            "attribute_index"
        ]
    )
)

print(f"Rows:              {len(experience_scores):,}")
print(f"Titles:            {experience_scores['title_key'].nunique():,}")
print(f"Experiences:       {experience_scores['experience'].nunique():,}")
print(
    "Duplicate title/experience rows:",
    experience_scores.duplicated(
        subset=["title_key", "experience"]
    ).sum()
)

Rows:              424,655
Titles:            7,721
Experiences:       55
Duplicate title/experience rows: 0


### Create relative experience scores

Raw cosine similarities are useful for preserving the model output, but their ranges can differ across experience attributes. Rank titles within each experience to create a more interpretable 0–100 score.

A score of 90 means that a title ranks higher than about 90% of titles in this dataset for that experience. It does not mean the title is "90% comforting," "90% intense," or any other literal percentage.

In [16]:
# Convert similarity into a 0-100 relative score for each experience
experience_scores["score"] = (
    experience_scores
    .groupby("experience")["raw_similarity"]
    .rank(
        method="average",
        pct=True
    )
    .mul(100)
    .round(2)
)

experience_scores = experience_scores[
    [
        "title_key",
        "tmdb_id",
        "media_type",
        "experience",
        "category",
        "score",
        "raw_similarity",
        "reviews_used"
    ]
]

experience_scores.head()

,title_key,tmdb_id,media_type,experience,category,score,raw_similarity,reviews_used
0,movie:100,100,movie,Happy,Emotion,53.34,0.457023,574
1,movie:10003,10003,movie,Happy,Emotion,43.50,0.451529,195
2,movie:100042,100042,movie,Happy,Emotion,46.25,0.452890,224
3,movie:10007,10007,movie,Happy,Emotion,20.30,0.438150,192
4,movie:10008,10008,movie,Happy,Emotion,32.22,0.445112,758


### Sanity check the experience profiles

Inspect the highest-scoring experiences for *The Dark Knight* to make sure the final title profile is reasonably coherent.

In [17]:
experience_scores[
    experience_scores["title_key"] == "movie:155"
].sort_values(
    "score",
    ascending=False
).head(10)[
    [
        "experience",
        "category",
        "score",
        "raw_similarity",
        "reviews_used"
    ]
]

,experience,category,score,raw_similarity,reviews_used
372164,Unforgettable,After-effect,87.76,0.556148,6815
86487,Full of life,Overall experience,78.95,0.514096,6815
240907,Epic / grand,Story / world experience,73.40,0.520530,6815
109650,Life-affirming,Overall experience,69.95,0.501892,6815
294954,Strong chemistry,Connection,62.23,0.543995,6815
125092,Immersive,Overall experience,61.05,0.555413,6815
387606,Fulfilled,After-effect,51.70,0.510933,6815
279512,Human / humanistic,Connection,49.36,0.508101,6815
140534,Existential,Thinking / engagement,49.23,0.476952,6815
101929,Intense,Overall experience,48.85,0.515592,6815


## Save and validate the final scores

Save the title-experience dataset as Parquet and run final checks before loading it into the analytical database.

In [18]:
experience_output_dir = (
    project_root
    / "data"
    / "processed"
    / "experience_scores"
)

experience_output_dir.mkdir(
    parents=True,
    exist_ok=True
)

experience_output_file = (
    experience_output_dir
    / "experience_scores.parquet"
)

experience_scores.to_parquet(
    experience_output_file,
    index=False
)

print(f"Saved: {experience_output_file}")
print(f"Rows: {len(experience_scores):,}")
print(f"Titles: {experience_scores['title_key'].nunique():,}")
print(f"Experiences: {experience_scores['experience'].nunique():,}")

Saved: C:\Users\dorca\OneDrive\Desktop\movies-tv-data-project\data\processed\experience_scores\experience_scores.parquet
Rows: 424,655
Titles: 7,721
Experiences: 55


In [19]:
print(
    "Duplicate title/experience rows:",
    experience_scores.duplicated(
        subset=["title_key", "experience"]
    ).sum()
)

print(
    "Missing scores:",
    experience_scores["score"].isna().sum()
)

print(
    "Score range:",
    experience_scores["score"].min(),
    "to",
    experience_scores["score"].max()
)

Duplicate title/experience rows: 0
Missing scores: 0
Score range: 0.01 to 100.0


## Result

The final dataset contains 424,655 title-experience records: 55 experience scores for each of 7,721 movies and TV shows, derived from 2,723,221 cleaned audience reviews.

Each score is a percentile rank within its experience, making it possible to compare titles based on how audiences describe the viewing experience rather than genre alone.

The scores are semantic signals rather than ground-truth labels. Some experience attributes overlap and some matches remain noisy, so the profiles are intended for relative ranking and recommendation rather than strict classification.